# AFX in scikit-learn Pipelines

`afx.sklearn.AFXFeatureExtractor` is a standard sklearn transformer: it turns audio files **or** raw signals into fixed-length feature matrices, so AFX drops directly into `Pipeline`, `cross_val_score`, and `GridSearchCV`.


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

from afx_audio.sklearn import AFXFeatureExtractor
from afx_audio.io.io import load_audio

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, GridSearchCV

SR = 16000

## Build a labeled dataset

With `ESC_50_dataset` pointing to a local [ESC-50](https://github.com/karolpiczak/ESC-50) checkout, two of its categories are used. Otherwise, including when only Nonspeech7k or UrbanSound8K is set (their metadata differs), the notebook synthesizes clips of two acoustic classes.

In [ ]:
import glob

# Environmental-sound dataset roots are passed via environment variables.
# Set any ONE of these before launching Jupyter, e.g.
#   export ESC_50_dataset=/path/to/ESC-50
DATASET_ENV_VARS = ['dataset_nonspeech7k', 'nonspeech7k_dataset',
                    'ESC_50_dataset', 'UrbanSound8K_dataset']

def find_dataset_audio(max_files=None, exts=('.wav', '.ogg', '.flac')):
    """Return audio file paths from the first configured dataset env var
    (empty list if no dataset is configured)."""
    for var in DATASET_ENV_VARS:
        root = os.environ.get(var)
        if root and os.path.isdir(root):
            files = sorted(
                p for ext in exts for p in
                glob.glob(os.path.join(root, '**', '*' + ext), recursive=True)
            )
            if files:
                print(f'Dataset: {var} -> {root} ({len(files)} audio files)')
            return files[:max_files] if max_files else files
    return []

def synth_clip(kind, seconds=1.0, sr=22050, seed=0):
    """Synthesize a simple environmental-sound-like clip (fallback when no
    dataset env var is configured)."""
    rng = np.random.default_rng(seed)
    t = np.arange(int(sr * seconds)) / sr
    if kind == 'tonal':        # bird-chirp-like rising tone
        f = 800 + 600 * t
        sig = 0.6 * np.sin(2 * np.pi * f * t)
    elif kind == 'noisy':      # rain-like broadband noise
        sig = 0.4 * rng.standard_normal(t.size)
    else:                      # engine-like low harmonic stack
        sig = sum(0.3 / k * np.sin(2 * np.pi * 90 * k * t) for k in (1, 2, 3))
    sig = sig + 0.02 * rng.standard_normal(t.size)
    return sig.astype(np.float32)

def build_dataset(n_per_class=16):
    esc = os.environ.get('ESC_50_dataset')
    meta = os.path.join(esc, 'meta', 'esc50.csv') if esc else ''
    if esc and os.path.isfile(meta):
        import pandas as pd
        df = pd.read_csv(meta)
        cats = ['dog', 'rain']
        X, y = [], []
        for label, cat in enumerate(cats):
            rows = df[df.category == cat].head(n_per_class)
            for fn in rows.filename:
                X.append(os.path.join(esc, 'audio', fn))
                y.append(label)
        print(f'ESC-50: {len(X)} files from categories {cats}')
        return X, np.array(y), cats
    print('Using synthetic two-class audio (set ESC_50_dataset for real data).')
    X = [synth_clip('tonal', sr=SR, seed=i) for i in range(n_per_class)] + \
        [synth_clip('noisy', sr=SR, seed=i) for i in range(n_per_class)]
    y = np.array([0] * n_per_class + [1] * n_per_class)
    return X, y, ['tonal', 'noisy']

X, y, class_names = build_dataset()
print('samples:', len(X), '- classes:', class_names)

## Extract features, sklearn-style

`fit` learns the output schema (feature names / dimensionality); `transform` produces the `(n_samples, n_features)` matrix. Inputs can be file paths or raw 1-D signals interchangeably.

In [ ]:
extractor = AFXFeatureExtractor(
    features=['mfcc', 'spectral_centroid', 'spectral_flatness',
              'zcr', 'rms_energy'],
    sample_rate=SR,
    aggregation='mean',
    n_jobs=1,          # set to -1 to parallelize over all cores via joblib
)
F = extractor.fit_transform(X)
names = extractor.get_feature_names_out()
print('feature matrix:', F.shape)
print('first columns:', list(names[:6]), '...')

## Full pipeline with cross-validation

Scaling belongs in the pipeline (`StandardScaler`), not in the extractor: per-sample `normalize` options exist but would normalize each clip independently.

In [ ]:
pipe = make_pipeline(
    AFXFeatureExtractor(features=['mfcc', 'spectral_centroid',
                                  'spectral_flatness', 'zcr', 'rms_energy'],
                        sample_rate=SR),
    StandardScaler(),
    LogisticRegression(max_iter=1000),
)
scores = cross_val_score(pipe, X, y, cv=4)
print(f'4-fold accuracy: {scores.mean():.2%} +/- {scores.std():.2%}')

## Hyperparameter search over extraction settings

Because the extractor is a standard estimator, extraction choices (aggregation, feature set, ...) are searchable like any other hyperparameter.

In [ ]:
param_grid = {
    'afxfeatureextractor__aggregation': ['mean', 'summary'],
    'afxfeatureextractor__features': [
        ['mfcc'],
        ['mfcc', 'spectral_centroid', 'spectral_flatness'],
    ],
}
search = GridSearchCV(pipe, param_grid, cv=2)
search.fit(X, y)
print('best params:', search.best_params_)
print(f'best CV accuracy: {search.best_score_:.2%}')

## Which features matter?

`get_feature_names_out()` keeps model inspection readable.

In [ ]:
ext = AFXFeatureExtractor(
    features=['mfcc', 'spectral_centroid', 'spectral_flatness',
              'zcr', 'rms_energy'], sample_rate=SR)
F = ext.fit_transform(X)
forest = RandomForestClassifier(n_estimators=200, random_state=0).fit(F, y)

imp = forest.feature_importances_
top = np.argsort(imp)[::-1][:10]
plt.figure(figsize=(7, 3.2))
plt.bar(range(len(top)), imp[top])
plt.xticks(range(len(top)), [ext.get_feature_names_out()[i] for i in top],
           rotation=45, ha='right')
plt.ylabel('importance')
plt.title(f'Top features for {class_names[0]} vs {class_names[1]}')
plt.tight_layout()

## Notes

`n_jobs=-1` parallelizes extraction over samples with joblib. Any config from `afx_audio/config.json` can be passed as `config` to control frame sizes and hops. For variable-length audio use a pooling aggregation (`mean`, `summary`, ...); `stack` needs equal-length inputs.